# Drug Development Landscape
## Part 2: Relational Database Creation with SQLite
This notebook converts the structured datasets obtained from the ClinicalTrials.gov API into a normalized SQLite relational database.

The database is designed to reduce redundancy and establish relationships between clinical studies, sponsors, interventions, disease conditions, and locations, enabling efficient SQL-based analysis.

## Import Libraries
Import the libraries required to create and interact with the SQLite database.

In [2]:
import sqlite3
import pandas as pd
import os

## Load Extracted DataFrames
The structured datasets generated during the data collection stage are used as the input for database creation.

In [3]:
studies_df = pd.read_csv("../data/studies.csv")

sponsors_df = pd.read_csv("../data/sponsors.csv")

interventions_df = pd.read_csv("../data/interventions.csv")

conditions_df = pd.read_csv("../data/conditions.csv")

locations_df = pd.read_csv("../data/locations.csv")

## Create SQLite Database Connection
A SQLite database was created to store the extracted clinical trial information in a relational format.

In [4]:
connection = sqlite3.connect(
    "../database/drug_development_landscape.db"
)

## Create Studies Table
The studies table stores the main characteristics of each clinical trial, including study identifier, title, recruitment status, clinical phase, and enrollment.

In [5]:
studies_df.to_sql(
    "studies",
    connection,
    if_exists="replace",
    index=False
)

5000

In [6]:
pd.read_sql(
    "SELECT * FROM studies LIMIT 5;",
    connection
)

,study_id,title,status,phase,enrollment
0,NCT01042717,Study of the Best Timing for Plerixafor in Aut...,UNKNOWN,NaN,10.0
1,NCT04941040,Opioid Free VS Opioid Anesthesia for Craniotomies,COMPLETED,PHASE1,60.0
2,NCT05883540,Lysergic Acid Diethylamide (LSD) in Palliative...,RECRUITING,PHASE2,60.0
3,NCT00171717,Conversion From Tacrolimus to Cyclosporine Mic...,COMPLETED,PHASE4,39.0
4,NCT00022217,Cisplatin-Epinephrine Injectable Gel Plus Pacl...,UNKNOWN,PHASE2,NaN


## Create Sponsors Table
Sponsors are separated into a unique lookup table to avoid repeating the same sponsor information across multiple studies. A unique identifier was assigned to each sponsor to establish relationships between studies and sponsors.

In [7]:
sponsors_df.head()

,study_id,sponsor_name,sponsor_type
0,NCT01042717,"Shi, Patricia, M.D.",INDIV
1,NCT01042717,"Genzyme, a Sanofi Company",INDUSTRY
2,NCT04941040,Kasr El Aini Hospital,OTHER
3,NCT05883540,"University Hospital, Basel, Switzerland",OTHER
4,NCT05883540,"University Hospital, Zürich",OTHER


In [8]:
sponsors_unique = sponsors_df[
    ["sponsor_name", "sponsor_type"]
].drop_duplicates()

In [9]:
sponsors_unique["sponsor_id"] = sponsors_unique.index + 1

## Create Study-Sponsor Relationship Table
Because a clinical trial can have multiple sponsors and a sponsor can participate in multiple studies, a many-to-many relationship table was created.

In [10]:
study_sponsors = sponsors_df.merge(
    sponsors_unique,
    on=["sponsor_name", "sponsor_type"],
    how="left"
)

In [11]:
study_sponsors.head(10)

,study_id,sponsor_name,sponsor_type,sponsor_id
0,NCT01042717,"Shi, Patricia, M.D.",INDIV,1
1,NCT01042717,"Genzyme, a Sanofi Company",INDUSTRY,2
2,NCT04941040,Kasr El Aini Hospital,OTHER,3
3,NCT05883540,"University Hospital, Basel, Switzerland",OTHER,4
4,NCT05883540,"University Hospital, Zürich",OTHER,5
5,NCT05883540,"Spital Uster AG, Uster, Switzerland",UNKNOWN,6
6,NCT05883540,"University Hospital, Geneva",OTHER,7
7,NCT00171717,Novartis,INDUSTRY,8
8,NCT00022217,Matrix Pharmaceutical,INDUSTRY,9
9,NCT01714089,Revalesio Corporation,INDUSTRY,10


In [12]:
study_sponsors = study_sponsors[
    ["study_id", "sponsor_id"]
]

In [13]:
study_sponsors.head()

,study_id,sponsor_id
0,NCT01042717,1
1,NCT01042717,2
2,NCT04941040,3
3,NCT05883540,4
4,NCT05883540,5


In [14]:
sponsors_unique.to_sql(
    "sponsors",
    connection,
    if_exists="replace",
    index=False
)

study_sponsors.to_sql(
    "study_sponsors",
    connection,
    if_exists="replace",
    index=False
)

7590

In [15]:
pd.read_sql(
    "SELECT * FROM sponsors LIMIT 5;",
    connection
)

,sponsor_name,sponsor_type,sponsor_id
0,"Shi, Patricia, M.D.",INDIV,1
1,"Genzyme, a Sanofi Company",INDUSTRY,2
2,Kasr El Aini Hospital,OTHER,3
3,"University Hospital, Basel, Switzerland",OTHER,4
4,"University Hospital, Zürich",OTHER,5


In [16]:
pd.read_sql(
    "SELECT * FROM study_sponsors LIMIT 5;",
    connection
)

,study_id,sponsor_id
0,NCT01042717,1
1,NCT01042717,2
2,NCT04941040,3
3,NCT05883540,4
4,NCT05883540,5


## Create Interventions Table
Interventions were stored as an independent table containing unique drugs and intervention types.

In [17]:
interventions_unique = interventions_df[
    ["intervention_name", "intervention_type"]
].drop_duplicates()

In [18]:
interventions_unique = interventions_unique.reset_index(drop=True)

In [19]:
interventions_unique["intervention_id"] = interventions_unique.index + 1

## Create Study-Intervention Relationship Table
A bridge table was created because a study can evaluate multiple interventions, and the same intervention can appear in multiple studies.

In [20]:
study_interventions = interventions_df.merge(
    interventions_unique,
    on=["intervention_name", "intervention_type"],
    how="left"
)

In [21]:
study_interventions = study_interventions[
    ["study_id", "intervention_id"]
]

In [22]:
interventions_unique.to_sql(
    "interventions",
    connection,
    if_exists="replace",
    index=False
)

study_interventions.to_sql(
    "study_interventions",
    connection,
    if_exists="replace",
    index=False
)

11015

In [23]:
study_interventions.head()

,study_id,intervention_id
0,NCT01042717,1
1,NCT04941040,2
2,NCT04941040,3
3,NCT05883540,4
4,NCT00171717,5


In [24]:
pd.read_sql(
    "SELECT * FROM interventions LIMIT 5;",
    connection
)

,intervention_name,intervention_type,intervention_id
0,Plerixafor,DRUG,1
1,Opioid free anesthetics,DRUG,2
2,Opioid Anesthetics,DRUG,3
3,Lysergic Acid Diethylamide Tartrate,DRUG,4
4,Cyclosporine - cyclosporine microemulsion,DRUG,5


## Create Conditions Table
Disease conditions were normalized into a separate table to avoid repeated disease names across studies.

In [25]:
conditions_unique = conditions_df[
    ["condition"]
].drop_duplicates()

In [26]:
conditions_unique = conditions_unique.reset_index(drop=True)

In [27]:
conditions_unique["condition_id"] = conditions_unique.index + 1

## Create Study-Condition Relationship Table
A relationship table connects clinical studies with their associated disease conditions.

In [28]:
study_conditions = conditions_df.merge(
    conditions_unique,
    on=["condition"],
    how="left"
)

In [29]:
conditions_unique.to_sql(
    "conditions",
    connection,
    if_exists="replace",
    index=False
)

study_conditions.to_sql(
    "study_conditions",
    connection,
    if_exists="replace",
    index=False
)

8549

In [30]:
pd.read_sql(
    "SELECT * FROM conditions LIMIT 5;",
    connection
)

,condition,condition_id
0,Multiple Myeloma,1
1,Non-Hodgkins Lymphoma,2
2,Supratentorial Neoplasms,3
3,Palliative Care,4
4,Pain,5


## Create Locations Table
Research facilities were normalized into a separate lookup table to avoid storing duplicate location information across multiple clinical studies. Each location was assigned a unique identifier (location_id).

In [32]:
locations_df.head()

,study_id,facility,city,country
0,NCT01042717,Mount Sinai School of Medicine,New York,United States
1,NCT04941040,Kasr El Aini Hospital,Cairo,Egypt
2,NCT05883540,"University Hospital Basel, Division of Clinica...",Basel,Switzerland
3,NCT05883540,"University Hospital Geneva, Palliative medicin...",Collonge-Bellerive,Switzerland
4,NCT05883540,"Spital Uster AG, Division of Internal Medicine",Uster,Switzerland


In [33]:
locations_unique = locations_df[
    ["facility","city","country"]
].drop_duplicates()

In [34]:
locations_unique = locations_unique.reset_index(drop=True)

In [35]:
locations_unique["location_id"] = locations_unique.index + 1

In [38]:
locations_unique.to_sql(
    "locations",
    connection,
    if_exists="replace",
    index=False
)

37941

In [39]:
pd.read_sql(
    """
    SELECT *
    FROM locations
    LIMIT 5;
    """,
    connection
)

,facility,city,country,location_id
0,Mount Sinai School of Medicine,New York,United States,1
1,Kasr El Aini Hospital,Cairo,Egypt,2
2,"University Hospital Basel, Division of Clinica...",Basel,Switzerland,3
3,"University Hospital Geneva, Palliative medicin...",Collonge-Bellerive,Switzerland,4
4,"Spital Uster AG, Division of Internal Medicine",Uster,Switzerland,5


## Create Study-Location Relationship Table
A relationship table was created to link clinical studies with their corresponding research locations. This design allows a single study to be associated with multiple research centers while preventing repeated storage of location information.

In [41]:
study_locations = locations_df.merge(
    locations_unique,
    on=["facility", "city","country"],
    how="left"
)

In [42]:
study_locations = study_locations[
    ["study_id", "location_id"]
]

In [43]:
study_locations.to_sql(
    "study_locations",
    connection,
    if_exists="replace",
    index=False
)

57896

In [44]:
study_locations.head()

,study_id,location_id
0,NCT01042717,1
1,NCT04941040,2
2,NCT05883540,3
3,NCT05883540,4
4,NCT05883540,5


## Database Verification
The final database contains normalized tables connected through relational keys.

In [40]:
pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    connection
)

,name
0,studies
1,sponsors
2,study_sponsors
3,interventions
4,study_interventions
5,conditions
6,study_conditions
7,locations


## Summary
A normalized SQLite database was successfully created from the ClinicalTrials.gov datasets.

The database structure separates independent entities (studies, sponsors, interventions, conditions, and locations) and uses relationship tables to represent many-to-many connections.

This structure enables flexible SQL queries for clinical trial landscape analysis, performed in the next notebook.